##### Step 1: Convert CSV to JSON Files

In [ ]:
import csv
import json
import uuid
import os

raw_reviews_file = "../data/raw/hotel_reviews_1000.csv"
transformed_dir = "../data/transformed"

raw_reviews = open(raw_reviews_file, "r").readlines()

if not os.path.exists(transformed_dir):
    os.makedirs(transformed_dir)

def process_reviews(file_path):
    with open(file_path, 'r', newline='', encoding='utf-8') as csvfile:
        # Read the first line to get the header
        header = next(csv.reader(csvfile))
        
        # Create a mapping of expected column names to actual column names
        column_mapping = {
            'dateAdded': 'dateAdded',
            'city': 'city',
            'hotel_name': 'name',
            'hotel_state': 'province',
            'review_text': 'reviews.text',
            'review_title': 'reviews.title'
        }
        
        # Find the index of each required column
        column_indices = {}
        for expected_name, actual_name in column_mapping.items():
            try:
                column_indices[expected_name] = header.index(actual_name)
            except ValueError:
                print(f"Warning: Column '{actual_name}' not found in the CSV. Some data may be missing.")
        
        # Reset file pointer to the beginning
        csvfile.seek(0)
        
        # Skip the header row
        next(csvfile)
        
        # Use csv.reader instead of DictReader
        reader = csv.reader(csvfile)
        
        for i, row in enumerate(reader, start=1):
            review_json = {}
            for key, index in column_indices.items():
                if index < len(row):
                    review_json[key] = row[index]
                else:
                    review_json[key] = ""  # or None, depending on your preference
            
            # Generate a unique identifier
            review_json['id'] = str(uuid.uuid4())
            
            # print(json.dumps(review_json, indent=2))
            print(f"processed record [{i}] with id [{review_json['id']}]")

            with open(f"{transformed_dir}/review_{i}.json", "w+") as f:
                json.dump(review_json, f, indent=2)
            
process_reviews(raw_reviews_file)

#### Step 2: Create Embeddings for each of the JSON Files

In [ ]:
%pip install -q python-dotenv openai

In [2]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)

response = client.embeddings.create(
    input="Hello world",
    model=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"],
)

print(len(response.data[0].embedding))
print(response.data[0].embedding)
print(response)

1536
[-0.0021610260009765625, -0.049072265625, 0.0209808349609375, 0.0313720703125, -0.045318603515625, -0.0264129638671875, -0.028961181640625, 0.060302734375, -0.0257110595703125, -0.0148162841796875, 0.015472412109375, -0.0300445556640625, -0.0203704833984375, -0.033416748046875, 0.0258331298828125, 0.01422119140625, -0.070068359375, 0.0123748779296875, 0.01483917236328125, 0.048919677734375, 0.020782470703125, -0.00887298583984375, -0.0151214599609375, -0.016632080078125, 0.0259552001953125, -0.002857208251953125, -0.024383544921875, 0.0242919921875, 0.0018329620361328125, -0.05572509765625, 0.023101806640625, -0.04547119140625, -0.00870513916015625, 0.0030956268310546875, 0.004558563232421875, 0.0018224716186523438, 0.0266876220703125, 0.01013946533203125, -0.01201629638671875, -0.0115203857421875, -0.01491546630859375, -0.0231475830078125, 0.025390625, 0.03680419921875, -0.035491943359375, 0.02130126953125, -0.0631103515625, 0.040374755859375, 0.053558349609375, 0.06158447265625,

In [4]:
import os
import json

transformed_dir = "../data/transformed"
embedded_dir = "../data/embedded"

os.makedirs(embedded_dir, exist_ok=True)

def prepare_embedding_str(review_json):
    return (
        f"REVIEW_TITLE: {review_json['review_title']} "
        f"REVIEW_TEXT: {review_json['review_text']} "
        f"HOTEL_NAME: {review_json['hotel_name']} "
        f"HOTEL_CITY: {review_json['city']} "
        f"HOTEL_STATE: {review_json['hotel_state']}"
    )

# Reuse the AzureOpenAI client created in the previous cell.
for file in os.listdir(transformed_dir):
    if not file.endswith(".json"):
        continue

    with open(os.path.join(transformed_dir, file), "r", encoding="utf-8") as f:
        review = json.load(f)

    embedding_str = prepare_embedding_str(review)

    response = client.embeddings.create(
        input=embedding_str,
        model=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"],
    )

    review["embedding"] = response.data[0].embedding

    with open(os.path.join(embedded_dir, file), "w", encoding="utf-8") as f:
        json.dump(review, f, indent=2)